<a href="https://colab.research.google.com/github/ST3ALT4/ucs547/blob/main/Assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numba
print(numba.__version__)

0.60.0


# Question 1

In [2]:
import numpy as np
import math
import time

from numba import cuda

N = 5000000

x_float32 = np.random.rand(N).astype(np.float32)

start_cpu = time.time()

cpu_result32 = x_float32**2 + 3*x_float32 + 5

end_cpu = time.time()

cpu_time32 = end_cpu - start_cpu

@cuda.jit
def compute_kernel32(input_array, output_array):

    idx = cuda.grid(1)

    if idx < input_array.size:
        x = input_array[idx]
        output_array[idx] = x*x + 3*x + 5

gpu_result32 = np.zeros_like(x_float32)

d_input32 = cuda.to_device(x_float32)
d_output32 = cuda.device_array_like(gpu_result32)

threads_per_block = 256
blocks_per_grid = math.ceil(N / threads_per_block)

start_gpu = time.time()

compute_kernel32[blocks_per_grid, threads_per_block](
    d_input32,
    d_output32
)

cuda.synchronize()

end_gpu = time.time()

gpu_time32 = end_gpu - start_gpu

gpu_result32 = d_output32.copy_to_host()

print("FLOAT32 RESULTS")
print("----------------")
print("CPU Time :", cpu_time32, "seconds")
print("GPU Time :", gpu_time32, "seconds")

x_float64 = np.random.rand(N).astype(np.float64)

start_cpu = time.time()

cpu_result64 = x_float64**2 + 3*x_float64 + 5

end_cpu = time.time()

cpu_time64 = end_cpu - start_cpu

@cuda.jit
def compute_kernel64(input_array, output_array):

    idx = cuda.grid(1)

    if idx < input_array.size:
        x = input_array[idx]
        output_array[idx] = x*x + 3*x + 5

gpu_result64 = np.zeros_like(x_float64)

d_input64 = cuda.to_device(x_float64)
d_output64 = cuda.device_array_like(gpu_result64)

start_gpu = time.time()

compute_kernel64[blocks_per_grid, threads_per_block](
    d_input64,
    d_output64
)

cuda.synchronize()

end_gpu = time.time()

gpu_time64 = end_gpu - start_gpu

gpu_result64 = d_output64.copy_to_host()

print("\nFLOAT64 RESULTS")
print("----------------")
print("CPU Time :", cpu_time64, "seconds")
print("GPU Time :", gpu_time64, "seconds")

FLOAT32 RESULTS
----------------
CPU Time : 0.020473003387451172 seconds
GPU Time : 3.7467243671417236 seconds

FLOAT64 RESULTS
----------------
CPU Time : 0.0406031608581543 seconds
GPU Time : 0.1489241123199463 seconds


# Question 2

In [3]:
import numpy as np
import time

from numba import njit

N = 1000000

data = np.random.randint(0, 100, N)

bins = 100

def python_histogram(data, bins):

    hist = [0] * bins

    for value in data:
        hist[value] += 1

    return hist

def numpy_histogram(data, bins):

    hist, _ = np.histogram(data, bins=bins, range=(0, bins))

    return hist

@njit
def numba_histogram(data, bins):

    hist = np.zeros(bins, dtype=np.int64)

    for i in range(len(data)):
        hist[data[i]] += 1

    return hist

start = time.time()

hist_python = python_histogram(data, bins)

end = time.time()

python_time = end - start

print("Pure Python Time :", python_time, "seconds")

start = time.time()

hist_numpy = numpy_histogram(data, bins)

end = time.time()

numpy_time = end - start

print("NumPy Time :", numpy_time, "seconds")

numba_histogram(data, bins)

start = time.time()

hist_numba = numba_histogram(data, bins)

end = time.time()

numba_time = end - start

print("Numba Time :", numba_time, "seconds")

print("\nCorrectness Check")

print("Python vs NumPy :",
      np.array_equal(np.array(hist_python), hist_numpy))

print("NumPy vs Numba :",
      np.array_equal(hist_numpy, hist_numba))

Pure Python Time : 0.4207417964935303 seconds
NumPy Time : 0.04191470146179199 seconds
Numba Time : 0.0012803077697753906 seconds

Correctness Check
Python vs NumPy : True
NumPy vs Numba : True


# Question 3

In [4]:
import random
import time

from numba import njit

def monte_carlo_pi_python(nsamples):

    inside_circle = 0

    for _ in range(nsamples):

        x = random.random()
        y = random.random()

        if x*x + y*y < 1:
            inside_circle += 1

    pi_estimate = 4 * inside_circle / nsamples

    return pi_estimate

@njit
def monte_carlo_pi_numba(nsamples):

    inside_circle = 0

    for _ in range(nsamples):

        x = random.random()
        y = random.random()

        if x*x + y*y < 1:
            inside_circle += 1

    pi_estimate = 4 * inside_circle / nsamples

    return pi_estimate

N = 5000000

start = time.time()

pi_python = monte_carlo_pi_python(N)

end = time.time()

python_time = end - start

print("PURE PYTHON")
print("Estimated PI :", pi_python)
print("Execution Time :", python_time, "seconds")

monte_carlo_pi_numba(10)

start = time.time()

pi_numba = monte_carlo_pi_numba(N)

end = time.time()

numba_time = end - start

print("\nNUMBA")
print("Estimated PI :", pi_numba)
print("Execution Time :", numba_time, "seconds")

speedup = python_time / numba_time

print("\nSpeedup Factor :", speedup)

PURE PYTHON
Estimated PI : 3.1414752
Execution Time : 0.9155330657958984 seconds

NUMBA
Estimated PI : 3.141312
Execution Time : 0.052593231201171875 seconds

Speedup Factor : 17.407811706680206


# Question 4

In [5]:
import numpy as np
import time

from numba import vectorize

@vectorize(['int64(int64)'])
def adjust_brightness(pixel_value):

    new_value = int(pixel_value * 1.2)

    if new_value > 255:
        return 255

    return new_value

N = 10000000

pixels = np.random.randint(
            0,
            256,
            N,
            dtype=np.int64
         )

start = time.time()

bright_pixels = adjust_brightness(pixels)

end = time.time()

print("Execution Time :", end - start, "seconds")

print("\nFirst 10 Original Pixels:")
print(pixels[:10])

print("\nFirst 10 Brightened Pixels:")
print(bright_pixels[:10])

Execution Time : 0.023699045181274414 seconds

First 10 Original Pixels:
[ 18 224  74 100  88  11  42 225 105  78]

First 10 Brightened Pixels:
[ 21 255  88 120 105  13  50 255 126  93]


In [6]:
import numpy as np
import time

from numba import vectorize

@vectorize(['int64(int64)'], target='parallel')
def adjust_brightness_parallel(pixel_value):

    new_value = int(pixel_value * 1.2)

    if new_value > 255:
        return 255

    return new_value

N = 10000000

pixels = np.random.randint(
            0,
            256,
            N,
            dtype=np.int64
         )

start = time.time()

bright_pixels = adjust_brightness_parallel(pixels)

end = time.time()

print("Parallel Execution Time :",
      end - start,
      "seconds")

Parallel Execution Time : 0.020691871643066406 seconds


# Question 5

In [7]:
import numpy as np
import time

from numba import njit

samples = 100000
features = 10

np.random.seed(42)

X = np.random.randn(samples, features)

# Binary labels {-1, +1}
y = np.random.choice([-1, 1], size=samples)

learning_rate = 0.01
epochs = 100

def sigmoid(z):

    return 1 / (1 + np.exp(-z))

def logistic_regression_numpy(X, y):

    weights = np.zeros(X.shape[1])

    for _ in range(epochs):

        linear = np.dot(X, weights)

        predictions = sigmoid(linear)

        # Convert labels {-1,+1} to {0,1}
        y_binary = (y + 1) / 2

        gradient = np.dot(
                        X.T,
                        (predictions - y_binary)
                   ) / len(y)

        weights -= learning_rate * gradient

    return weights

@njit
def logistic_regression_numba(X, y,
                              learning_rate,
                              epochs):

    weights = np.zeros(X.shape[1])

    for _ in range(epochs):

        linear = np.dot(X, weights)

        predictions = 1 / (1 + np.exp(-linear))

        y_binary = (y + 1) / 2

        gradient = np.dot(
                        X.T,
                        (predictions - y_binary)
                   ) / len(y)

        weights -= learning_rate * gradient

    return weights

start = time.time()

weights_numpy = logistic_regression_numpy(X, y)

end = time.time()

numpy_time = end - start

print("NUMPY VERSION")
print("Execution Time :", numpy_time, "seconds")

logistic_regression_numba(
    X,
    y,
    learning_rate,
    1
)

start = time.time()

weights_numba = logistic_regression_numba(
                    X,
                    y,
                    learning_rate,
                    epochs
                )

end = time.time()

numba_time = end - start

print("\nNUMBA VERSION")
print("Execution Time :", numba_time, "seconds")

difference = np.linalg.norm(
                weights_numpy - weights_numba
             )

print("\nDifference Between Weights :",
      difference)

speedup = numpy_time / numba_time

print("\nSpeedup Factor :", speedup)

NUMPY VERSION
Execution Time : 0.2540452480316162 seconds

NUMBA VERSION
Execution Time : 0.5774919986724854 seconds

Difference Between Weights : 8.261485383671575e-19

Speedup Factor : 0.43991128641713634


# Question 6

In [8]:
import numpy as np
import math
import time

from numba import cuda

N = 1024


A = np.random.rand(N, N).astype(np.float32)
B = np.random.rand(N, N).astype(np.float32)

C = np.zeros((N, N), dtype=np.float32)

@cuda.jit
def matrix_add_kernel(A, B, C):

    row, col = cuda.grid(2)

    if row < C.shape[0] and col < C.shape[1]:

        C[row, col] = A[row, col] + B[row, col]

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array_like(C)

threads_per_block = (16, 16)

blocks_per_grid_x = math.ceil(N / threads_per_block[0])
blocks_per_grid_y = math.ceil(N / threads_per_block[1])

blocks_per_grid = (
    blocks_per_grid_x,
    blocks_per_grid_y
)

start = time.time()

matrix_add_kernel[
    blocks_per_grid,
    threads_per_block
](
    d_A,
    d_B,
    d_C
)

cuda.synchronize()

end = time.time()

C = d_C.copy_to_host()

print("Execution Time :",
      end - start,
      "seconds")

print("\nSample Output:")

print("A[0][0] =", A[0][0])
print("B[0][0] =", B[0][0])
print("C[0][0] =", C[0][0])

Execution Time : 0.13873624801635742 seconds

Sample Output:
A[0][0] = 0.48813277
B[0][0] = 0.9814017
C[0][0] = 1.4695344
